# 01 — Seniority EDA & Classifier

End-to-end: load data → explore → train → evaluate → save model.

In [ ]:
import sys, os, django
sys.path.insert(0, os.path.abspath('../..'))
os.environ.setdefault('DJANGO_SETTINGS_MODULE', 'config.settings.local')
django.setup()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from apps.intelligence.models import Job

df = pd.DataFrame(Job.objects.values('id', 'title', 'description', 'source', 'seniority'))
df['desc_len'] = df['description'].str.split().str.len().fillna(0).astype(int)
print(df.shape)
df['seniority'].value_counts()

In [ ]:
# Class distribution
df['seniority'].value_counts().plot(kind='bar', title='Seniority distribution')
plt.tight_layout()

In [ ]:
# Description length by class
df.boxplot(column='desc_len', by='seniority', figsize=(8, 4))
plt.suptitle('')
plt.title('Description word count by seniority')
plt.tight_layout()

In [ ]:
# Source x class cross-tab
pd.crosstab(df['source'], df['seniority'], margins=True)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import joblib, pathlib

df['text'] = df['title'] + ' ' + df['description']

X_train, X_test, y_train, y_test = train_test_split(
    df['text'], df['seniority'], test_size=0.2, stratify=df['seniority'], random_state=42
)

pipe = Pipeline([
    ('tfidf', TfidfVectorizer(max_features=10_000, ngram_range=(1, 2), sublinear_tf=True)),
    ('clf',   LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)),
])
pipe.fit(X_train, y_train)

print(classification_report(y_test, pipe.predict(X_test)))

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, pipe.predict(X_test))
plt.tight_layout()

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipe, df['text'], df['seniority'], cv=cv, scoring='f1_macro')
print(f'CV macro-F1: {scores.mean():.3f} ± {scores.std():.3f}')

In [ ]:
out = pathlib.Path('../../apps/intelligence/ml')
out.mkdir(exist_ok=True)
joblib.dump(pipe, out / 'seniority_classifier.joblib')
print('Saved:', out / 'seniority_classifier.joblib')